# Pruned QAT CNN to hls4ml for `cnn_small_hls_opt_img512`

This notebook is a linear, notebook-owned workflow for exploring pruned QKeras QAT models and hls4ml settings for the Coyote bitstream classifier.

Run it top-to-bottom to:

1. Build deterministic balanced k-fold splits from the dataset manifests.
2. Train a pruned + QAT QKeras CNN on fold 0 and inspect training/accuracy.
3. Check sparsity.
4. Convert/compile the fold-0 model with hls4ml, inspect the spec, plot the graph, and run bit-accurate emulation.
5. Run representative-fold logic synthesis.
6. Train the remaining folds and produce pooled k-fold plots/metrics for the newly trained notebook model.
7. Convert/emulate all folds and compare pooled QKeras vs hls4ml accuracy.

The notebook does not load old model artifacts. Long-running results are reused only when the current iteration fingerprint matches the saved manifest exactly.

## Imports

In [ ]:
%matplotlib inline

import csv
import hashlib
import json
import math
import os
import pprint
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

# QKeras 0.9 + hls4ml 1.3 in this workspace expects legacy tf.keras.
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PATH"] = os.path.expanduser("~/.local/graphviz/bin") + ":" + os.environ.get("PATH", "")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tensorflow_model_optimization.sparsity import keras as sparsity
from tensorflow_model_optimization.python.core.sparsity.keras import pruning_callbacks

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name != "hls4ml":
    raise RuntimeError(f"Run this notebook from the hls4ml workspace root, got {WORKSPACE}")
EXAMPLE_ROOT = WORKSPACE.parent
for path in (WORKSPACE, EXAMPLE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PY_CANDIDATES = [
    EXAMPLE_ROOT / ".venv_hls4ml" / "bin" / "python",
    EXAMPLE_ROOT / ".venv" / "bin" / "python",
    Path(sys.executable),
]
PY = next((path for path in PY_CANDIDATES if path.exists() and os.access(path, os.X_OK)), Path(sys.executable))

print("workspace:", WORKSPACE)
print("python:", PY)
print("tensorflow:", tf.__version__)
print("dot:", shutil.which("dot"))

## Editable Parameters

In [ ]:
# Iteration identity. Changing any parameter below also changes the fingerprinted artifact directory.
ITERATION_NAME = "pruned_qat_w6_a6_s50_rf8"

# Candidate/dataset settings.
CANDIDATE_NAME = "cnn_small_hls_opt_img512"
IMG_SIZE = 512
MIN_RO = 8000
K_FOLDS = 5
PRIMARY_FOLD = 0
SEED = 42
BALANCE_CLASSES = True

# Model architecture. This mirrors SmallCNNHlsOptimized512 by default.
INPUT_SHAPE = (512, 512, 1)
CONV_SPECS = [
    {"filters": 8, "kernel": (5, 5), "strides": (2, 2), "pad": 2, "name": "conv0"},
    {"filters": 16, "kernel": (3, 3), "strides": (1, 1), "pad": 1, "name": "conv1"},
    {"filters": 24, "kernel": (3, 3), "strides": (1, 1), "pad": 1, "name": "conv2"},
    {"filters": 24, "kernel": (3, 3), "strides": (1, 1), "pad": 1, "name": "conv3"},
    {"filters": 32, "kernel": (3, 3), "strides": (1, 1), "pad": 1, "name": "conv4"},
]
FINAL_AVG_POOL = (8, 8)
OUTPUT_UNITS = 1

# QKeras quantization settings.
QUANTIZER_TAG = "w6_a6"
WEIGHT_BITS = 6
WEIGHT_INTEGER = 0
ACTIVATION_BITS = 6
ACTIVATION_INTEGER = [2, 2, 3, 4, 5]
QUANTIZER_ALPHA = 1

# Pruning settings.
PRUNE_FINAL_SPARSITY = 0.50
PRUNE_BEGIN_EPOCH = 2
PRUNE_END_EPOCH = 10
PRUNE_FREQUENCY_EPOCHS = 1
PRUNE_OUTPUT_DENSE = False

# Training settings. Defaults match the successful run style but now train pruned+QAT.
EPOCHS = 300
BATCH_SIZE = 8
LR = 1e-4
AUGMENT = True
FLIP_H_PROB = 0.5
FLIP_V_PROB = 0.5
CROP_SCALE_MIN = 1.0
TRANSLATE = 0.0
CACHE_DATA = True

# hls4ml settings.
HLS_BACKEND = "Vitis"
HLS_IO_TYPE = "io_stream"
HLS_STRATEGY = "Resource"
HLS_REUSE_FACTOR = 8
HLS_CLOCK_PERIOD = 5.0
HLS_PART = "xcu55c-fsvh2892-2L-e"
HLS_OUTPUT_PRECISION = "fixed<16,6,RND,SAT>"
HLS_POOL_ACCUM_PRECISION = None
HLS_ACCUM_PRECISION = None
N_EMULATION_SAMPLES = None  # None means the full validation fold.
SYNTHESIS_FOLDS = [PRIMARY_FOLD]  # Representative-fold synthesis by default.

CONFIG = {
    "iteration_name": ITERATION_NAME,
    "candidate_name": CANDIDATE_NAME,
    "img_size": IMG_SIZE,
    "min_ro": MIN_RO,
    "k_folds": K_FOLDS,
    "primary_fold": PRIMARY_FOLD,
    "seed": SEED,
    "balance_classes": BALANCE_CLASSES,
    "input_shape": INPUT_SHAPE,
    "conv_specs": CONV_SPECS,
    "final_avg_pool": FINAL_AVG_POOL,
    "output_units": OUTPUT_UNITS,
    "quantizer_tag": QUANTIZER_TAG,
    "weight_bits": WEIGHT_BITS,
    "weight_integer": WEIGHT_INTEGER,
    "activation_bits": ACTIVATION_BITS,
    "activation_integer": ACTIVATION_INTEGER,
    "quantizer_alpha": QUANTIZER_ALPHA,
    "prune_final_sparsity": PRUNE_FINAL_SPARSITY,
    "prune_begin_epoch": PRUNE_BEGIN_EPOCH,
    "prune_end_epoch": PRUNE_END_EPOCH,
    "prune_frequency_epochs": PRUNE_FREQUENCY_EPOCHS,
    "prune_output_dense": PRUNE_OUTPUT_DENSE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "augment": AUGMENT,
    "flip_h_prob": FLIP_H_PROB,
    "flip_v_prob": FLIP_V_PROB,
    "crop_scale_min": CROP_SCALE_MIN,
    "translate": TRANSLATE,
    "cache_data": CACHE_DATA,
    "hls_backend": HLS_BACKEND,
    "hls_io_type": HLS_IO_TYPE,
    "hls_strategy": HLS_STRATEGY,
    "hls_reuse_factor": HLS_REUSE_FACTOR,
    "hls_clock_period": HLS_CLOCK_PERIOD,
    "hls_part": HLS_PART,
    "hls_output_precision": HLS_OUTPUT_PRECISION,
    "hls_pool_accum_precision": HLS_POOL_ACCUM_PRECISION,
    "hls_accum_precision": HLS_ACCUM_PRECISION,
    "n_emulation_samples": N_EMULATION_SAMPLES,
    "synthesis_folds": SYNTHESIS_FOLDS,
}
CONFIG

## Helpers: Fingerprints, Dataset Splits, and Artifacts

In [ ]:
from dataset import bitstream_to_array, load_manifest
from train import compute_metrics_from_outputs
from pipeline.evaluation import write_metrics_summary
from pipeline.qkeras_plots import (
    build_run_params,
    build_split_info,
    history_rows_to_columns,
    load_final_metrics,
    write_fold_plots,
    write_kfold_plots,
)
from pipeline.qkeras_qat import (
    BitstreamKerasSequence,
    QATTrainConfig,
    apply_numpy_augmentation,
    history_row,
    metrics_from_logits,
    predict_sequence,
    qkeras_per_sample_rows,
    sample_to_nhwc,
    write_history,
    write_qkeras_per_sample,
)

SOURCE_FILES_FOR_FINGERPRINT = [
    EXAMPLE_ROOT / "dataset.py",
    EXAMPLE_ROOT / "model.py",
    EXAMPLE_ROOT / "train.py",
    WORKSPACE / "pipeline" / "qkeras_qat.py",
    WORKSPACE / "pipeline" / "qkeras_plots.py",
]


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_json(payload) -> str:
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)


SOURCE_HASHES = {str(path.relative_to(WORKSPACE.parent)): file_sha256(path) for path in SOURCE_FILES_FOR_FINGERPRINT if path.exists()}
FINGERPRINT_PAYLOAD = {"config": CONFIG, "source_hashes": SOURCE_HASHES}
FINGERPRINT = hashlib.sha256(canonical_json(FINGERPRINT_PAYLOAD).encode()).hexdigest()
SHORT_FINGERPRINT = FINGERPRINT[:12]

ITERATION_ROOT = WORKSPACE / "artifacts" / CANDIDATE_NAME / "notebook_pruned_qat" / f"{ITERATION_NAME}_{SHORT_FINGERPRINT}"
ITERATION_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = ITERATION_ROOT / "iteration_manifest.json"

manifest = {
    "fingerprint": FINGERPRINT,
    "short_fingerprint": SHORT_FINGERPRINT,
    "created_or_reused_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": CONFIG,
    "source_hashes": SOURCE_HASHES,
}
if MANIFEST_PATH.exists():
    old_manifest = json.loads(MANIFEST_PATH.read_text())
    if old_manifest.get("fingerprint") != FINGERPRINT:
        raise RuntimeError(f"Fingerprint collision/stale manifest in {ITERATION_ROOT}")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True))

print("iteration root:", ITERATION_ROOT)
print("fingerprint:", FINGERPRINT)

In [ ]:
def load_balanced_samples_from_manifest(min_ro: int, seed: int, balance: bool) -> list[dict]:
    samples = load_manifest(min_ro=min_ro)
    labels = [int(s["class_label"]) for s in samples]
    n_benign = sum(label == 0 for label in labels)
    n_stand = sum(label == 1 for label in labels)
    if not balance or n_benign <= n_stand:
        return samples
    rng = np.random.RandomState(seed)
    benign_samples = [s for s in samples if int(s["class_label"]) == 0]
    stand_samples = [s for s in samples if int(s["class_label"]) == 1]
    benign_keep = rng.choice(len(benign_samples), size=n_stand, replace=False)
    benign_samples = [benign_samples[i] for i in sorted(benign_keep)]
    return benign_samples + stand_samples


def make_kfold_splits(samples: list[dict], k_folds: int, seed: int) -> list[tuple[list[dict], list[dict]]]:
    labels = np.asarray([int(s["class_label"]) for s in samples])
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    splits = []
    for train_idx, val_idx in skf.split(samples, labels):
        splits.append(([samples[i] for i in train_idx], [samples[i] for i in val_idx]))
    return splits


def class_counts(rows: list[dict]) -> dict[int, int]:
    return {label: sum(int(row["class_label"]) == label for row in rows) for label in (0, 1)}


samples = load_balanced_samples_from_manifest(MIN_RO, SEED, BALANCE_CLASSES)
splits = make_kfold_splits(samples, K_FOLDS, SEED)

split_rows = []
for fold, (train_samples, val_samples) in enumerate(splits):
    split_rows.append({
        "fold": fold,
        "n_train": len(train_samples),
        "train_benign": class_counts(train_samples)[0],
        "train_standalone": class_counts(train_samples)[1],
        "n_val": len(val_samples),
        "val_benign": class_counts(val_samples)[0],
        "val_standalone": class_counts(val_samples)[1],
    })

split_df = pd.DataFrame(split_rows)
display(split_df)
print("pooled samples:", len(samples), class_counts(samples))

# Persist the exact split for auditability.
split_dir = ITERATION_ROOT / "splits"
split_dir.mkdir(exist_ok=True)
for fold, (train_samples, val_samples) in enumerate(splits):
    for name, rows in [("train", train_samples), ("val", val_samples)]:
        with (split_dir / f"fold_{fold}_{name}.csv").open("w", newline="") as handle:
            fieldnames = sorted({key for row in rows for key in row.keys()})
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

## Helpers: QKeras Model, Pruning, Training, and Metrics

In [ ]:
from qkeras import QActivation, QConv2D, QDense
from qkeras.utils import _add_supported_quantized_objects


def weight_quantizer() -> str:
    return f"quantized_bits({WEIGHT_BITS},{WEIGHT_INTEGER},alpha={QUANTIZER_ALPHA})"


def activation_quantizer(block_idx: int) -> str:
    return f"quantized_relu({ACTIVATION_BITS},{ACTIVATION_INTEGER[block_idx]})"


def build_qkeras_notebook_model(name_suffix: str = ""):
    layers = tf.keras.layers
    models = tf.keras.models
    x = x_in = layers.Input(shape=INPUT_SHAPE, name="bitstream_input")
    for i, spec in enumerate(CONV_SPECS):
        x = layers.ZeroPadding2D(padding=spec["pad"], name=f"pad_{spec['name']}")(x)
        x = QConv2D(
            spec["filters"],
            kernel_size=spec["kernel"],
            strides=spec["strides"],
            padding="valid",
            kernel_quantizer=weight_quantizer(),
            bias_quantizer=weight_quantizer(),
            kernel_initializer="lecun_uniform",
            use_bias=True,
            name=spec["name"],
        )(x)
        x = QActivation(activation_quantizer(i), name=f"act{i}")(x)
        x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), name=f"pool{i}")(x)
    x = layers.AveragePooling2D(pool_size=FINAL_AVG_POOL, strides=FINAL_AVG_POOL, name="gap")(x)
    x = layers.Flatten(name="flatten")(x)
    x = QDense(
        OUTPUT_UNITS,
        kernel_quantizer=weight_quantizer(),
        bias_quantizer=weight_quantizer(),
        kernel_initializer="lecun_uniform",
        use_bias=True,
        name="output_dense",
    )(x)
    return models.Model(inputs=[x_in], outputs=[x], name=f"qkeras_pruned_qat_{QUANTIZER_TAG}{name_suffix}")


qmodel_template = build_qkeras_notebook_model()
qmodel_template.summary()

try:
    from qkeras.autoqkeras.utils import print_qmodel_summary
    print_qmodel_summary(qmodel_template)
except Exception as exc:
    print("QKeras quantizer summary unavailable:", exc)

In [ ]:
class KerasSequenceAdapter(tf.keras.utils.Sequence):
    def __init__(self, seq, **kwargs):
        super().__init__(**kwargs)
        self.seq = seq

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return self.seq[idx]

    def on_epoch_end(self):
        if hasattr(self.seq, "on_epoch_end"):
            self.seq.on_epoch_end()


def notebook_train_config(fold: int) -> QATTrainConfig:
    return QATTrainConfig(
        candidate_name=CANDIDATE_NAME,
        quantizer_tag=QUANTIZER_TAG,
        fold=fold,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        seed=SEED,
        augment=AUGMENT,
        flip_h_prob=FLIP_H_PROB,
        flip_v_prob=FLIP_V_PROB,
        crop_scale_min=CROP_SCALE_MIN,
        translate=TRANSLATE,
        cache_data=CACHE_DATA,
        gradcam=False,
    )


@dataclass(frozen=True)
class NotebookCandidate:
    name: str = CANDIDATE_NAME
    model: str = CANDIDATE_NAME
    representation: str = "2d"
    img_size: int = IMG_SIZE
    sequence_length: int = IMG_SIZE * IMG_SIZE
    min_ro: int = MIN_RO
    target_part: str = HLS_PART
    folds: tuple[int, ...] = tuple(range(K_FOLDS))


candidate = NotebookCandidate()


def make_notebook_sequences(fold: int):
    train_samples, val_samples = splits[fold]
    cfg = notebook_train_config(fold)
    train_seq = BitstreamKerasSequence(train_samples, candidate, cfg, shuffle=True, augment=AUGMENT)
    val_seq = BitstreamKerasSequence(val_samples, candidate, cfg, shuffle=False, augment=False)
    aug_val_seq = BitstreamKerasSequence(val_samples, candidate, cfg, shuffle=False, augment=AUGMENT)
    return cfg, train_seq, val_seq, aug_val_seq, train_samples, val_samples


def prune_function_factory(nsteps: int):
    pruning_params = {
        "pruning_schedule": sparsity.PolynomialDecay(
            initial_sparsity=0.0,
            final_sparsity=PRUNE_FINAL_SPARSITY,
            begin_step=nsteps * PRUNE_BEGIN_EPOCH,
            end_step=nsteps * PRUNE_END_EPOCH,
            frequency=max(1, nsteps * PRUNE_FREQUENCY_EPOCHS),
        )
    }

    def prune_function(layer):
        if layer.name == "output_dense" and not PRUNE_OUTPUT_DENSE:
            return layer
        if layer.__class__.__name__ in {"Conv2D", "Dense", "QConv2D", "QDense"}:
            return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params)
        return layer

    return prune_function


class EpochMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_seq, aug_val_seq):
        super().__init__()
        self.val_seq = val_seq
        self.aug_val_seq = aug_val_seq
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_labels, val_logits = predict_sequence(self.model, self.val_seq)
        aug_labels, aug_logits = predict_sequence(self.model, self.aug_val_seq)
        val_metrics = metrics_from_logits(val_labels, val_logits)
        aug_metrics = metrics_from_logits(aug_labels, aug_logits)
        row = history_row(epoch + 1, float(logs.get("loss", float("nan"))), val_metrics, aug_metrics)
        self.rows.append(row)
        print(
            f" metrics: val_acc={val_metrics['accuracy']:.4f} "
            f"val_pr_auc={val_metrics['pr_auc']:.4f} aug_acc={aug_metrics['accuracy']:.4f}"
        )


def fold_dir(fold: int) -> Path:
    return ITERATION_ROOT / f"fold_{fold}"


def fold_manifest_path(fold: int) -> Path:
    return fold_dir(fold) / "training_manifest.json"


def fold_cache_valid(fold: int) -> bool:
    fdir = fold_dir(fold)
    manifest_path = fold_manifest_path(fold)
    required = [
        manifest_path,
        fdir / "final_weights.weights.h5",
        fdir / "history.csv",
        fdir / "per_sample.csv",
        fdir / "metrics_summary.json",
    ]
    if not all(path.exists() for path in required):
        return False
    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception:
        return False
    return manifest.get("fingerprint") == FINGERPRINT and manifest.get("fold") == fold


def load_fold_model(fold: int):
    model = build_qkeras_notebook_model(name_suffix=f"_fold{fold}")
    model.load_weights(fold_dir(fold) / "final_weights.weights.h5")
    return model


def train_or_load_fold(fold: int):
    fdir = fold_dir(fold)
    fdir.mkdir(parents=True, exist_ok=True)
    cfg, train_seq, val_seq, aug_val_seq, train_samples, val_samples = make_notebook_sequences(fold)

    if fold_cache_valid(fold):
        print(f"Fold {fold}: exact cache hit at {fdir}")
        model = load_fold_model(fold)
        metrics = load_final_metrics(fdir, "per_sample.csv")
        aug_metrics = load_final_metrics(fdir, "augmented_per_sample.csv")
        history_rows = list(csv.DictReader((fdir / "history.csv").open()))
        return {
            "fold": fold,
            "out_dir": fdir,
            "model": model,
            "metrics": metrics,
            "aug_metrics": aug_metrics,
            "history_columns": history_rows_to_columns(history_rows),
            "train_samples": train_samples,
            "val_samples": val_samples,
            "final_epoch": EPOCHS,
        }

    print(f"Fold {fold}: training pruned QAT model")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED + fold)
    nsteps = math.ceil(len(train_samples) / BATCH_SIZE)
    base_model = build_qkeras_notebook_model(name_suffix=f"_fold{fold}")
    pruned_model = tf.keras.models.clone_model(base_model, clone_function=prune_function_factory(nsteps))
    pruned_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        run_eagerly=True,
    )

    metrics_cb = EpochMetricsCallback(val_seq, aug_val_seq)
    callbacks = [pruning_callbacks.UpdatePruningStep(), metrics_cb]
    pruned_model.fit(
        KerasSequenceAdapter(train_seq),
        epochs=EPOCHS,
        validation_data=KerasSequenceAdapter(val_seq),
        callbacks=callbacks,
        verbose=1,
    )

    stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
    val_labels, val_logits = predict_sequence(stripped_model, val_seq)
    aug_labels, aug_logits = predict_sequence(stripped_model, aug_val_seq)
    val_metrics = metrics_from_logits(val_labels, val_logits)
    aug_metrics = metrics_from_logits(aug_labels, aug_logits)

    write_history(fdir / "history.csv", metrics_cb.rows)
    write_metrics_summary(
        fdir / "metrics_summary.json",
        val_metrics,
        extra={"fingerprint": FINGERPRINT, "fold": fold, "stage": "pruned_qat", "candidate": CANDIDATE_NAME},
    )
    write_metrics_summary(
        fdir / "augmented_metrics_summary.json",
        aug_metrics,
        extra={"fingerprint": FINGERPRINT, "fold": fold, "stage": "pruned_qat_augmented", "candidate": CANDIDATE_NAME},
    )
    per_sample_rows = write_qkeras_per_sample(fdir / "per_sample.csv", val_samples, val_labels, val_logits)
    write_qkeras_per_sample(fdir / "augmented_per_sample.csv", val_samples, aug_labels, aug_logits)
    stripped_model.save_weights(fdir / "final_weights.weights.h5")
    (fdir / "model_config.json").write_text(stripped_model.to_json())

    training_manifest = {
        "fingerprint": FINGERPRINT,
        "fold": fold,
        "candidate": CANDIDATE_NAME,
        "config": CONFIG,
        "n_train": len(train_samples),
        "n_val": len(val_samples),
        "train_counts": class_counts(train_samples),
        "val_counts": class_counts(val_samples),
    }
    fold_manifest_path(fold).write_text(json.dumps(training_manifest, indent=2, sort_keys=True))

    history_columns = history_rows_to_columns(metrics_cb.rows)
    split_info = build_split_info(CANDIDATE_NAME, fold, len(train_samples), len(val_samples))
    write_fold_plots(
        fdir,
        history_columns,
        val_metrics,
        aug_metrics,
        split_info=split_info,
        run_params={"iteration": ITERATION_NAME, "fingerprint": SHORT_FINGERPRINT, "prune": PRUNE_FINAL_SPARSITY, "quantizer": QUANTIZER_TAG},
        final_epoch=EPOCHS,
    )

    return {
        "fold": fold,
        "out_dir": fdir,
        "model": stripped_model,
        "metrics": val_metrics,
        "aug_metrics": aug_metrics,
        "history_columns": history_columns,
        "train_samples": train_samples,
        "val_samples": val_samples,
        "final_epoch": EPOCHS,
    }

## Check Unroll Limit

In [ ]:
UNROLL_LIMIT = 4096
unroll_rows = []
for layer in qmodel_template.layers:
    if layer.__class__.__name__ in {"QConv2D", "Conv2D", "QDense", "Dense"}:
        weights = layer.get_weights()
        n_weights = int(np.prod(weights[0].shape)) if weights else 0
        unroll_rows.append({
            "layer": layer.name,
            "class": layer.__class__.__name__,
            "weights": n_weights,
            "over_4096": n_weights > UNROLL_LIMIT,
        })
unroll_df = pd.DataFrame(unroll_rows)
display(unroll_df)
if unroll_df["over_4096"].any():
    print("Not all layers fit the 4096-weight full-unroll rule; Resource strategy/reuse sweeps are appropriate.")

## Train Primary Fold and Plot Training/Accuracy

In [ ]:
primary_result = train_or_load_fold(PRIMARY_FOLD)
trained_qkeras = primary_result["model"]

primary_metrics = primary_result["metrics"]
metric_keys = ["accuracy", "balanced_accuracy", "roc_auc", "pr_auc", "bce_loss", "optimal_accuracy", "optimal_threshold"]
display(pd.DataFrame([{key: primary_metrics.get(key) for key in metric_keys}], index=[f"fold_{PRIMARY_FOLD}"]))

for path in [
    fold_dir(PRIMARY_FOLD) / "training_curves.png",
    fold_dir(PRIMARY_FOLD) / "evaluation_dashboard.png",
    fold_dir(PRIMARY_FOLD) / "final_evaluation_plots.png",
]:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

## Check Sparsity

In [ ]:
def weight_sparsity(model):
    rows = []
    weights_by_layer = []
    labels = []
    for layer in model.layers:
        ws = layer.get_weights()
        if not ws:
            continue
        w = np.asarray(ws[0]).reshape(-1)
        rows.append({
            "layer": layer.name,
            "class": layer.__class__.__name__,
            "n_weights": int(w.size),
            "zero_fraction": float(np.mean(w == 0.0)),
            "min": float(np.min(w)),
            "max": float(np.max(w)),
        })
        weights_by_layer.append(w)
        labels.append(layer.name)
    return pd.DataFrame(rows), weights_by_layer, labels

sparsity_df, weights_by_layer, labels = weight_sparsity(trained_qkeras)
display(sparsity_df)

fig = plt.figure(figsize=(10, 6))
plt.hist(weights_by_layer, bins=np.linspace(-1.2, 1.2, 80), histtype="stepfilled", stacked=True, label=labels, alpha=0.75)
plt.legend(fontsize=8)
plt.xlabel("Weight")
plt.ylabel("Count")
plt.title(f"Pruned QAT Weights, Fold {PRIMARY_FOLD}")
fig.tight_layout()
display(fig)
plt.close(fig)

## hls4ml Compile, Spec, and Plot for Primary Fold

In [ ]:
def qkeras_hls_config_for_model(model) -> dict:
    import hls4ml
    import keras

    keras_version = keras.__version__
    keras.__version__ = "2.15.0"
    try:
        config = hls4ml.utils.config_from_keras_model(model, granularity="name", backend=HLS_BACKEND)
    finally:
        keras.__version__ = keras_version

    config.setdefault("Model", {})
    config["Model"]["Strategy"] = HLS_STRATEGY
    config["Model"]["ReuseFactor"] = HLS_REUSE_FACTOR
    for layer in config.get("LayerName", {}).values():
        layer["ReuseFactor"] = HLS_REUSE_FACTOR
        precision = layer.get("Precision")
        if HLS_ACCUM_PRECISION and isinstance(precision, dict) and "accum" in precision:
            precision["accum"] = HLS_ACCUM_PRECISION
    if "output_dense" in config.get("LayerName", {}):
        config["LayerName"]["output_dense"].setdefault("Precision", {})["result"] = HLS_OUTPUT_PRECISION
    if "gap" in config.get("LayerName", {}):
        config["LayerName"]["gap"].setdefault("Precision", {})["accum"] = HLS_POOL_ACCUM_PRECISION
    return config


def hls_dir(fold: int) -> Path:
    return fold_dir(fold) / "hls" / f"qkeras_{QUANTIZER_TAG}_rf{HLS_REUSE_FACTOR}"


def compile_hls_for_fold(fold: int, model):
    import hls4ml
    import keras

    out_dir = hls_dir(fold)
    out_dir.mkdir(parents=True, exist_ok=True)
    project_name = f"{CANDIDATE_NAME}_{ITERATION_NAME}_fold{fold}_hls"
    manifest_path = out_dir / "conversion_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("fingerprint") == FINGERPRINT and (out_dir / "hls4ml_config.yml").exists():
            print(f"Fold {fold}: hls4ml exact cache hit at {out_dir}")
    config = qkeras_hls_config_for_model(model)
    keras_version = keras.__version__
    keras.__version__ = "2.15.0"
    try:
        hls_model = hls4ml.converters.convert_from_keras_model(
            model,
            hls_config=config,
            output_dir=str(out_dir),
            project_name=project_name,
            backend=HLS_BACKEND,
            io_type=HLS_IO_TYPE,
            part=HLS_PART,
            clock_period=HLS_CLOCK_PERIOD,
        )
    finally:
        keras.__version__ = keras_version
    hls_model.compile()
    manifest_path.write_text(json.dumps({
        "fingerprint": FINGERPRINT,
        "fold": fold,
        "project_name": project_name,
        "config": CONFIG,
        "hls_dir": str(out_dir),
    }, indent=2, sort_keys=True))
    return hls_model, config, out_dir


hls_model_primary, hls_config_primary, hls_primary_dir = compile_hls_for_fold(PRIMARY_FOLD, trained_qkeras)
print("hls dir:", hls_primary_dir)
print("Model config:")
pprint.pp(hls_config_primary.get("Model", {}))

print("\nFull hls4ml configuration:")
try:
    from hls4ml.utils import plotting
    plotting.print_dict(hls_config_primary)
except Exception:
    pprint.pp(hls_config_primary)

print("\nCompiled hls4ml model summary:")
try:
    hls_model_primary.summary()
except Exception as exc:
    print("summary unavailable:", exc)

layer_rows = []
for name, layer_cfg in hls_config_primary.get("LayerName", {}).items():
    precision = layer_cfg.get("Precision", {})
    layer_rows.append({
        "layer": name,
        "reuse_factor": layer_cfg.get("ReuseFactor"),
        "result_precision": precision.get("result") if isinstance(precision, dict) else precision,
        "accum_precision": precision.get("accum") if isinstance(precision, dict) else None,
        "weight_precision": precision.get("weight") if isinstance(precision, dict) else None,
    })
display(pd.DataFrame(layer_rows))

In [ ]:
import hls4ml
from hls4ml.model.profiling import numerical

try:
    numerical(model=trained_qkeras, hls_model=hls_model_primary)
except Exception as exc:
    print("numerical profiling failed:", exc)

plot_path = hls_primary_dir / "hls4ml_model.png"
try:
    hls4ml.utils.plot_model(
        hls_model_primary,
        show_shapes=True,
        show_precision=True,
        to_file=str(plot_path),
    )
    display(Image(filename=str(plot_path)))
    print(plot_path)
except Exception as exc:
    print("plot_model failed:", exc)
    print("pydot/dot check:")
    try:
        import pydot
        print("pydot", pydot.__version__, "dot", shutil.which("dot"))
        pydot.Dot().create(format="png")
    except Exception as pydot_exc:
        print("pydot smoke test failed:", pydot_exc)
    print("Inspect YAML instead:", hls_primary_dir / "hls4ml_config.yml")

## Bit-Accurate Emulation for Primary Fold

In [ ]:
def sigmoid(logits):
    logits = np.asarray(logits, dtype=np.float64)
    out = np.empty_like(logits)
    pos = logits >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-logits[pos]))
    exp_x = np.exp(logits[~pos])
    out[~pos] = exp_x / (1.0 + exp_x)
    return out


def validation_arrays_for_fold(fold: int):
    _, val_samples = splits[fold]
    xs = [sample_to_nhwc(sample, candidate) for sample in val_samples]
    ys = np.asarray([int(sample["class_label"]) for sample in val_samples], dtype=np.int32)
    x = np.stack(xs).astype(np.float32)
    if N_EMULATION_SAMPLES is not None:
        x = x[:N_EMULATION_SAMPLES]
        ys = ys[:N_EMULATION_SAMPLES]
        val_samples = val_samples[:N_EMULATION_SAMPLES]
    return x, ys, val_samples


def binary_log_loss(label: int, prob: float) -> float:
    p = min(max(float(prob), 1e-7), 1.0 - 1e-7)
    return -math.log(p) if int(label) == 1 else -math.log(1.0 - p)


def rows_from_logits(samples, labels, logits):
    probs = sigmoid(np.asarray(logits).reshape(-1))
    rows = []
    for idx, (sample, label, logit, prob) in enumerate(zip(samples, labels, logits, probs)):
        pred = int(prob >= 0.5)
        loss = binary_log_loss(int(label), float(prob))
        rows.append({
            "sample_index": idx,
            "sample_id": sample.get("sample_id", ""),
            "app_name": sample.get("app_name", ""),
            "class_label": int(label),
            "class_name": sample.get("class_name", "standalone" if int(label) else "benign"),
            "ro_count": sample.get("ro_count", ""),
            "bitstream_path": sample.get("bitstream_path", ""),
            "logit": f"{float(logit):.9f}",
            "probability": f"{float(prob):.9f}",
            "predicted_label": pred,
            "correct": pred == int(label),
            "per_sample_bce_loss": f"{loss:.9f}",
            "per_sample_log_loss": f"{loss:.9f}",
        })
    return rows


def metrics_from_stage_rows(rows):
    labels = np.asarray([int(row["class_label"]) for row in rows], dtype=np.float32)
    probs = np.asarray([float(row["probability"]) for row in rows], dtype=np.float32)
    losses = np.asarray([float(row["per_sample_bce_loss"]) for row in rows], dtype=np.float32)
    return compute_metrics_from_outputs(float(np.mean(losses)), labels, probs)


def write_csv(path: Path, rows: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def emulate_fold(fold: int, model, hls_model):
    parity_dir = fold_dir(fold) / "parity"
    parity_dir.mkdir(parents=True, exist_ok=True)
    x, labels, val_samples = validation_arrays_for_fold(fold)
    keras_logits = np.asarray(model.predict(x, verbose=0)).reshape(-1)
    hls_logits = np.asarray(hls_model.predict(np.ascontiguousarray(x))).reshape(-1)
    abs_err = np.abs(hls_logits - keras_logits)
    parity_rows = []
    for idx, (label, k_logit, h_logit, err) in enumerate(zip(labels, keras_logits, hls_logits, abs_err)):
        parity_rows.append({
            "idx": idx,
            "label": int(label),
            "keras_logit": float(k_logit),
            "hls_logit": float(h_logit),
            "abs_err": float(err),
            "rel_err": float(err / max(abs(float(k_logit)), 1e-6)),
        })
    write_csv(parity_dir / "parity.csv", parity_rows)

    keras_rows = rows_from_logits(val_samples, labels, keras_logits)
    hls_rows = rows_from_logits(val_samples, labels, hls_logits)
    write_csv(parity_dir / "qkeras_per_sample.csv", keras_rows)
    write_csv(parity_dir / "hls_per_sample.csv", hls_rows)

    keras_metrics = metrics_from_stage_rows(keras_rows)
    hls_metrics = metrics_from_stage_rows(hls_rows)
    summary = {
        "fingerprint": FINGERPRINT,
        "fold": fold,
        "n": int(len(labels)),
        "logit_mae": float(abs_err.mean()),
        "logit_max_abs": float(abs_err.max()),
        "sign_mismatches": int(np.sum((keras_logits >= 0.0) != (hls_logits >= 0.0))),
        "qkeras_accuracy": float(keras_metrics["accuracy"]),
        "hls_accuracy": float(hls_metrics["accuracy"]),
        "qkeras_pr_auc": float(keras_metrics["pr_auc"]),
        "hls_pr_auc": float(hls_metrics["pr_auc"]),
    }
    (parity_dir / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
    return summary, keras_rows, hls_rows

primary_parity_summary, primary_qkeras_rows, primary_hls_rows = emulate_fold(PRIMARY_FOLD, trained_qkeras, hls_model_primary)
display(pd.DataFrame([primary_parity_summary]))

## Representative Logic Synthesis

In [ ]:
def find_csynth_report(project_dir: Path):
    candidates = sorted(Path(project_dir).glob("*_prj/solution1/syn/report/*_csynth.rpt"))
    top = [p for p in candidates if "_hls_csynth" in p.name or p.name == "csynth.rpt"]
    return (top or candidates or [None])[0]


def parse_csynth_report(report_path: Path | None):
    if report_path is None or not Path(report_path).exists():
        return {}
    text = Path(report_path).read_text(errors="ignore").splitlines()
    out = {"report": str(report_path)}
    for i, line in enumerate(text):
        if "Latency (cycles)" in line:
            for row in text[i + 1 : i + 10]:
                parts = [p.strip() for p in row.split("|")]
                nums = [p for p in parts if p.replace("-", "").replace(".", "").isdigit()]
                if len(nums) >= 2:
                    out["latency_min_cycles"] = int(float(nums[0]))
                    out["latency_max_cycles"] = int(float(nums[1]))
                    break
        if "Interval (cycles)" in line:
            for row in text[i + 1 : i + 10]:
                parts = [p.strip() for p in row.split("|")]
                nums = [p for p in parts if p.replace("-", "").replace(".", "").isdigit()]
                if nums:
                    out["interval_cycles"] = int(float(nums[0]))
                    break
    return out


def synthesize_fold_if_needed(fold: int):
    project_dir = hls_dir(fold)
    synth_manifest = project_dir / "synthesis_manifest.json"
    report = find_csynth_report(project_dir)
    if synth_manifest.exists() and report is not None:
        manifest = json.loads(synth_manifest.read_text())
        if manifest.get("fingerprint") == FINGERPRINT:
            print(f"Fold {fold}: synthesis exact cache hit")
            row = {"fold": fold, "project_dir": str(project_dir), "cached": True}
            row.update(parse_csynth_report(report))
            return row

    if shutil.which("vitis_hls") is None:
        raise RuntimeError("vitis_hls is not on PATH. Run `source /opt/hdev/cli/enable/vitis` in the Jupyter server environment and restart the kernel/server.")
    if not (project_dir / "build_prj.tcl").exists():
        raise FileNotFoundError(f"Missing build_prj.tcl in {project_dir}")

    subprocess.run(["vitis_hls", "-f", "build_prj.tcl"], cwd=project_dir, check=True)
    report = find_csynth_report(project_dir)
    synth_manifest.write_text(json.dumps({"fingerprint": FINGERPRINT, "fold": fold, "completed_at": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2, sort_keys=True))
    row = {"fold": fold, "project_dir": str(project_dir), "cached": False}
    row.update(parse_csynth_report(report))
    return row

synthesis_rows = [synthesize_fold_if_needed(fold) for fold in SYNTHESIS_FOLDS]
display(pd.DataFrame(synthesis_rows))

## Train Remaining Folds and Produce Pooled K-Fold Plots

In [ ]:
fold_results = {PRIMARY_FOLD: primary_result}
for fold in range(K_FOLDS):
    if fold == PRIMARY_FOLD:
        continue
    fold_results[fold] = train_or_load_fold(fold)

ordered_fold_results = [fold_results[fold] for fold in range(K_FOLDS)]
plot_payload = []
for result in ordered_fold_results:
    plot_payload.append({
        "fold_label": f"fold_{result['fold']}",
        "history": result["history_columns"],
        "final_metrics": result["metrics"],
        "final_aug_metrics": result.get("aug_metrics"),
        "final_epoch": result["final_epoch"],
    })

write_kfold_plots(
    ITERATION_ROOT,
    plot_payload,
    split_info=f"Candidate: {CANDIDATE_NAME} | Iteration: {ITERATION_NAME} | Fingerprint: {SHORT_FINGERPRINT}",
    run_params={"quantizer": QUANTIZER_TAG, "prune": PRUNE_FINAL_SPARSITY, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR},
)

for path in [
    ITERATION_ROOT / "kfold_training_curves.png",
    ITERATION_ROOT / "evaluation_dashboard.png",
    ITERATION_ROOT / "final_evaluation_plots.png",
]:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

## Convert and Emulate All Folds, Then Compare Pooled Accuracy

In [ ]:
all_qkeras_rows = []
all_hls_rows = []
parity_summaries = []

for fold in range(K_FOLDS):
    model = fold_results[fold]["model"]
    hls_model, _, _ = compile_hls_for_fold(fold, model)
    summary, q_rows, h_rows = emulate_fold(fold, model, hls_model)
    parity_summaries.append(summary)
    for row in q_rows:
        row = dict(row)
        row["fold"] = fold
        all_qkeras_rows.append(row)
    for row in h_rows:
        row = dict(row)
        row["fold"] = fold
        all_hls_rows.append(row)

pooled_dir = ITERATION_ROOT / "pooled_parity"
pooled_dir.mkdir(exist_ok=True)
write_csv(pooled_dir / "qkeras_per_sample.csv", all_qkeras_rows)
write_csv(pooled_dir / "hls_per_sample.csv", all_hls_rows)

qkeras_pooled_metrics = metrics_from_stage_rows(all_qkeras_rows)
hls_pooled_metrics = metrics_from_stage_rows(all_hls_rows)
write_metrics_summary(pooled_dir / "qkeras_metrics_summary.json", qkeras_pooled_metrics, extra={"fingerprint": FINGERPRINT, "stage": "pruned_qat_qkeras"})
write_metrics_summary(pooled_dir / "hls_metrics_summary.json", hls_pooled_metrics, extra={"fingerprint": FINGERPRINT, "stage": "pruned_qat_hls"})

comparison = pd.DataFrame([
    {"stage": "QKeras pruned QAT", "accuracy": qkeras_pooled_metrics["accuracy"], "balanced_accuracy": qkeras_pooled_metrics["balanced_accuracy"], "roc_auc": qkeras_pooled_metrics["roc_auc"], "pr_auc": qkeras_pooled_metrics["pr_auc"], "bce_loss": qkeras_pooled_metrics["bce_loss"]},
    {"stage": "hls4ml bit-accurate", "accuracy": hls_pooled_metrics["accuracy"], "balanced_accuracy": hls_pooled_metrics["balanced_accuracy"], "roc_auc": hls_pooled_metrics["roc_auc"], "pr_auc": hls_pooled_metrics["pr_auc"], "bce_loss": hls_pooled_metrics["bce_loss"]},
])
display(pd.DataFrame(parity_summaries))
display(comparison)

# Simple pooled score-distribution plot.
fig, ax = plt.subplots(figsize=(8, 5))
for label, rows in [("QKeras", all_qkeras_rows), ("hls4ml", all_hls_rows)]:
    df = pd.DataFrame(rows)
    probs = df["probability"].astype(float).to_numpy()
    y = df["class_label"].astype(int).to_numpy()
    ax.hist(probs[y == 0], bins=30, alpha=0.35, density=True, label=f"{label} benign")
    ax.hist(probs[y == 1], bins=30, alpha=0.35, density=True, histtype="step", linewidth=2, label=f"{label} standalone")
ax.set_xlabel("P(standalone)")
ax.set_ylabel("Density")
ax.legend(fontsize=8)
ax.set_title("Pooled QKeras vs hls4ml Scores")
fig.tight_layout()
plot_path = pooled_dir / "pooled_score_distribution.png"
fig.savefig(plot_path, dpi=160)
display(fig)
plt.close(fig)
print(plot_path)

## Iteration Outputs

All outputs for this notebook iteration are under the fingerprinted directory printed below. Change parameters in the editable configuration cell to create a new iteration directory and regenerate stale results automatically.

In [ ]:
print("Iteration root:", ITERATION_ROOT)
print("Fingerprint:", FINGERPRINT)
print("Manifest:", MANIFEST_PATH)